# Stage3 GPU smoke

1. Attach `stage3-gpu-smoke.zip` as a **private** Kaggle Dataset.
2. Enable a GPU and run these cells in order. No training weights are downloaded.
3. Download the displayed `stage3-gpu-result.zip` after PASS.

This is a one-step runtime test, not full training or a submission.
The input may be a ZIP or Kaggle's automatically extracted dataset.


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, json, shutil, tempfile, zipfile

INPUT = Path('/kaggle/input')
WORK_PARENT = Path('/kaggle/working')
# Set SOURCE explicitly only when several matching bundles are attached.
SOURCE = None
if SOURCE is None:
    archives = list(INPUT.rglob('stage3-gpu-smoke.zip'))
    directories = [p.parent for p in INPUT.rglob('bundle.json')
                   if (p.parent/'src/stage3_pipeline.py').is_file()
                   and (p.parent/'dataset/split_manifest.csv').is_file()]
    candidates = archives + directories
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one Stage3 bundle; set SOURCE to one of: {candidates}')
    SOURCE = candidates[0]
SOURCE = Path(SOURCE)
WORK = Path(tempfile.mkdtemp(prefix='stage3-gpu-', dir=WORK_PARENT))
if SOURCE.is_file():
    with zipfile.ZipFile(SOURCE) as z:
        for member in z.infolist():
            rel = PurePosixPath(member.filename)
            if rel.is_absolute() or '..' in rel.parts or '\\' in member.filename:
                raise ValueError('Unsafe ZIP member')
            if not (WORK/member.filename).resolve().is_relative_to(WORK.resolve()):
                raise ValueError('ZIP path escapes workspace')
        z.extractall(WORK)
else:
    shutil.copytree(SOURCE, WORK, dirs_exist_ok=True)
BUNDLE = json.loads((WORK/'bundle.json').read_text())
if BUNDLE.get('kind') != 'GPU_SMOKE_ONLY':
    raise ValueError('Not a Stage3 smoke bundle')
for relative, expected in BUNDLE['file_sha256'].items():
    path = (WORK/relative).resolve()
    if not path.is_relative_to(WORK.resolve()):
        raise ValueError('Manifest path escapes workspace')
    if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise ValueError(f'Bundle checksum mismatch: {relative}')
print('Verified bundle:', SOURCE)
print('Working directory:', WORK)


## Environment

Use the baseline versions recorded in `requirements.txt` when matching the evaluation environment.
Package installation is not automatic. If required, run the following in a separate cell:

```python
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(WORK/'requirements.txt')], check=True)
```

Restart the kernel after replacing packages, then rerun from the first cell.
This creates a fresh working directory and preserves previous runs.


In [ ]:
import sys, platform, importlib.metadata
import torch, torchvision, numpy, pandas, cv2
from IPython.display import FileLink, display

ENVIRONMENT = {
    'python': sys.version,
    'platform': platform.platform(),
    'torch': str(torch.__version__),
    'torchvision': str(torchvision.__version__),
    'numpy': numpy.__version__, 'pandas': pandas.__version__,
    'opencv': cv2.__version__,
    'cuda_available': torch.cuda.is_available(),
    'torch_cuda': torch.version.cuda,
    'gpu_names': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
}
print(json.dumps(ENVIRONMENT, indent=2))
if not ENVIRONMENT['cuda_available']:
    raise RuntimeError('Enable a Kaggle GPU and use a CUDA-enabled PyTorch installation.')
(WORK/'environment.json').write_text(json.dumps(ENVIRONMENT, indent=2))
print('Baseline requirements:')
print((WORK/'requirements.txt').read_text())


In [ ]:
import subprocess, time

def run_logged(arguments, name):
    started = time.monotonic()
    with (WORK/name).open('w', encoding='utf-8') as log:
        process = subprocess.Popen([sys.executable, *arguments], cwd=WORK,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
        returncode = process.wait()
    if returncode:
        raise RuntimeError(f'{name} failed with exit code {returncode}; retain {WORK/name}')
    return {'exit_code': returncode, 'seconds': time.monotonic()-started}

RUNS = {}
RUNS['tests'] = run_logged(['src/test_stage3_pipeline.py'], 'tests.log')
RUNS['audit'] = run_logged(['src/stage3_pipeline.py', 'audit', '--dataset-dir', 'dataset'], 'audit.log')
RUNS['smoke'] = run_logged(['src/stage3_pipeline.py', 'smoke', '--dataset-dir', 'dataset',
                          '--output-dir', 'smoke-run', '--device', 'cuda'], 'smoke.log')
(WORK/'execution.json').write_text(json.dumps(RUNS, indent=2))


In [ ]:
report = json.loads((WORK/'smoke-run/smoke_report.json').read_text())
predictions = pandas.read_csv(WORK/'smoke-run/predictions.csv')
if report.get('status') != 'PASS' or report.get('device') != 'cuda' or report.get('public_cuda_entrypoint_tested') is not True:
    raise RuntimeError('GPU/public entrypoint verification did not pass')
if list(predictions.columns) != ['ID','sample_index','accel_label','steer_label']:
    raise RuntimeError('Prediction schema mismatch')
if predictions['ID'].tolist() != ['SMOKE_S3']*4 or predictions.sample_index.tolist() != list(range(4)):
    raise RuntimeError('Missing or extra smoke frames')
RESULT = WORK/'stage3-gpu-result.zip'
files = ['environment.json','execution.json','bundle.json','tests.log','audit.log','smoke.log',
         'smoke-run/smoke_report.json','smoke-run/predictions.csv']
with zipfile.ZipFile(RESULT, 'x', compression=zipfile.ZIP_DEFLATED) as z:
    for relative in files:
        z.write(WORK/relative, relative)
print('GPU SMOKE PASS — not full training')
print(json.dumps(report, indent=2))
display(FileLink(str(RESULT)))
